# Datasets


In [2]:
import subprocess, sys

print("Installing packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "datasets",
    "pandas",
    "matplotlib",
    "seaborn",
    "tqdm",
], capture_output=True)

Installing packages...


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'datasets', 'pandas', 'matplotlib', 'seaborn', 'tqdm'], returncode=0, stdout=b'', stderr=b'')

In [3]:
import json
import os
import re
import gzip
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm import tqdm

OUT_DIR = Path("/content/medagent_data")
OUT_DIR.mkdir(exist_ok=True)

print("Imports ready")
print(f"   Output directory: {OUT_DIR}")

Imports ready
   Output directory: /content/medagent_data


In [4]:
!git clone https://github.com/chanzuckerberg/MedMentions

Cloning into 'MedMentions'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 74 (delta 0), reused 1 (delta 0), pack-reused 71 (from 1)
Receiving objects: 100% (74/74), 24.74 MiB | 13.47 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [5]:
!git clone https://github.com/abachaa/MedQuAD

Cloning into 'MedQuAD'...
remote: Enumerating objects: 11310, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 11310 (delta 7), reused 5 (delta 5), pack-reused 11300 (from 1)
Receiving objects: 100% (11310/11310), 11.01 MiB | 16.27 MiB/s, done.
Resolving deltas: 100% (6807/6807), done.


In [6]:
import json
from pathlib import Path

def load_jsonl(path):
    """Load a JSON Lines file — one JSON object per line."""
    data = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:  # skip blank lines
                data.append(json.loads(line))
    return data

# ── Load BC5CDR as JSONL ───────────────────────────────────────────────────────
bc5cdr = {}
bc5cdr["train"] = load_jsonl("/content/train.json")
bc5cdr["test"]  = load_jsonl("/content/test.json")
bc5cdr["valid"] = load_jsonl("/content/valid.json")

with open("/content/label.json") as f:
    bc5cdr["label"] = json.load(f)  # label.json is regular JSON

print(f"BC5CDR loaded")
print(f"   train : {len(bc5cdr['train']):,} samples")
print(f"   test  : {len(bc5cdr['test']):,} samples")
print(f"   valid : {len(bc5cdr['valid']):,} samples")
print(f"   labels: {bc5cdr['label']}")
print(f"\nFirst sample:")
print(json.dumps(bc5cdr["train"][0], indent=2))

BC5CDR loaded
   train : 5,228 samples
   test  : 5,865 samples
   valid : 5,330 samples
   labels: {'O': 0, 'B-Chemical': 1, 'B-Disease': 2, 'I-Disease': 3, 'I-Chemical': 4}

First sample:
{
  "tags": [
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0
  ],
  "tokens": [
    "Naloxone",
    "reverses",
    "the",
    "antihypertensive",
    "effect",
    "of",
    "clonidine",
    "."
  ]
}


In [7]:
# ── Inspect structure ─────────────────────────────────────────────────────────
print("=" * 60)
print("BC5CDR — DATA STRUCTURE")
print("=" * 60)

sample = bc5cdr["train"][0]
print("\nFirst training sample (raw):")
print(json.dumps(sample, indent=2)[:800])

print("\n" + "-" * 60)
print("Label map:")
print(json.dumps(bc5cdr["label"], indent=2))

# Detect the key names (different versions use different key names)
sample_keys = list(sample.keys())
print(f"\nKeys in each sample: {sample_keys}")

# Try to find token and tag keys
token_key = next((k for k in sample_keys if "token" in k.lower() or "word" in k.lower()), sample_keys[0])
tag_key   = next((k for k in sample_keys if "tag" in k.lower() or "label" in k.lower() or "ner" in k.lower()), sample_keys[-1])
print(f"   Token key: '{token_key}'")
print(f"   Tag key:   '{tag_key}'")

# Show a readable entity-highlighted example
print("\n" + "-" * 60)
print("Entity visualization (first sample):")
tokens = sample[token_key] if isinstance(sample[token_key][0], str) else sample[token_key]
tags   = sample[tag_key]

# Handle nested lists (some versions wrap in extra list)
if isinstance(tokens[0], list): tokens = tokens[0]
if isinstance(tags[0], list):   tags   = tags[0]

# Convert numeric tags to string labels if needed
label_map = bc5cdr["label"]
if isinstance(tags[0], int):
    inv_label = {v: k for k, v in label_map.items()}
    tags = [inv_label.get(t, str(t)) for t in tags]

highlighted = []
for tok, tag in zip(tokens, tags):
    if tag.startswith("B-") or tag.startswith("I-"):
        entity_type = tag[2:]
        highlighted.append(f"[{tok}|{entity_type}]")
    else:
        highlighted.append(tok)
print(" ".join(highlighted[:60]))

BC5CDR — DATA STRUCTURE

First training sample (raw):
{
  "tags": [
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0
  ],
  "tokens": [
    "Naloxone",
    "reverses",
    "the",
    "antihypertensive",
    "effect",
    "of",
    "clonidine",
    "."
  ]
}

------------------------------------------------------------
Label map:
{
  "O": 0,
  "B-Chemical": 1,
  "B-Disease": 2,
  "I-Disease": 3,
  "I-Chemical": 4
}

Keys in each sample: ['tags', 'tokens']
   Token key: 'tokens'
   Tag key:   'tags'

------------------------------------------------------------
Entity visualization (first sample):
[Naloxone|Chemical] reverses the antihypertensive effect of [clonidine|Chemical] .


In [8]:
# ── Statistics ────────────────────────────────────────────────────────────────
print("BC5CDR — STATISTICS")
print("=" * 60)

for split_name, split_data in [("train", bc5cdr["train"]),
                                ("valid", bc5cdr["valid"]),
                                ("test",  bc5cdr["test"])]:
    tag_counts = Counter()
    total_tokens = 0

    for sample in split_data:
        tags = sample[tag_key]
        if isinstance(tags[0], list): tags = tags[0]
        if isinstance(tags[0], int):
            inv_label = {v: k for k, v in label_map.items()}
            tags = [inv_label.get(t, str(t)) for t in tags]
        tag_counts.update(tags)
        total_tokens += len(tags)

    chemicals = sum(v for k, v in tag_counts.items() if "Chemical" in k and k.startswith("B-"))
    diseases  = sum(v for k, v in tag_counts.items() if "Disease" in k and k.startswith("B-"))

    print(f"\n  {split_name.upper()}")
    print(f"    Sentences   : {len(split_data):,}")
    print(f"    Total tokens: {total_tokens:,}")
    print(f"    B-Chemical  : {chemicals:,}")
    print(f"    B-Disease   : {diseases:,}")
    print(f"    All tags    : {dict(tag_counts.most_common())}")

# ── Save unified format ───────────────────────────────────────────────────────
bc5cdr_out = {
    "dataset": "bc5cdr",
    "task": "NER",
    "entity_types": ["Chemical", "Disease"],
    "label_map": label_map,
    "token_key": token_key,
    "tag_key": tag_key,
    "splits": {
        "train": len(bc5cdr["train"]),
        "valid": len(bc5cdr["valid"]),
        "test":  len(bc5cdr["test"]),
    }
}
with open(OUT_DIR / "bc5cdr_meta.json", "w") as f:
    json.dump(bc5cdr_out, f, indent=2)

print("\nBC5CDR metadata saved to medagent_data/bc5cdr_meta.json")

BC5CDR — STATISTICS

  TRAIN
    Sentences   : 5,228
    Total tokens: 109,322
    B-Chemical  : 5,203
    B-Disease   : 4,182
    All tags    : {'O': 96796, 'B-Chemical': 5203, 'B-Disease': 4182, 'I-Disease': 2570, 'I-Chemical': 571}

  VALID
    Sentences   : 5,330
    Total tokens: 108,958
    B-Chemical  : 5,347
    B-Disease   : 4,244
    All tags    : {'O': 96413, 'B-Chemical': 5347, 'B-Disease': 4244, 'I-Disease': 2416, 'I-Chemical': 538}

  TEST
    Sentences   : 5,865
    Total tokens: 116,318
    B-Chemical  : 5,385
    B-Disease   : 4,424
    All tags    : {'O': 103684, 'B-Chemical': 5385, 'B-Disease': 4424, 'I-Disease': 2424, 'I-Chemical': 401}

BC5CDR metadata saved to medagent_data/bc5cdr_meta.json


In [12]:
import gzip, re, json
from pathlib import Path
from tqdm import tqdm

# ── Paths (exact from your file tree) ─────────────────────────────────────────
mm_path  = "/content/MedMentions/full/data/corpus_pubtator.txt.gz"
MM_PMIDS    = {
    "train": "/content/MedMentions/full/data/corpus_pubtator_pmids_trng.txt",
    "dev":   "/content/MedMentions/full/data/corpus_pubtator_pmids_dev.txt",
    "test":  "/content/MedMentions/full/data/corpus_pubtator_pmids_test.txt",
}

# ── Load split PMIDs ───────────────────────────────────────────────────────────
split_pmids = {}
for split, path in MM_PMIDS.items():
    with open(path) as f:
        split_pmids[split] = set(line.strip() for line in f if line.strip())
    print(f"  {split:6s}: {len(split_pmids[split]):,} PMIDs")

# ── Parse corpus ───────────────────────────────────────────────────────────────
def parse_pubtator_gz(gz_path):
    docs, current = [], None
    with gzip.open(gz_path, "rt", encoding="utf-8", errors="replace") as f:
        for line in tqdm(f, desc="Parsing", mininterval=2.0):
            line = line.rstrip("\n")
            if not line.strip():
                if current: docs.append(current); current = None
                continue
            m = re.match(r"^(\d+)\|t\|(.*)$", line)
            if m:
                current = {"pmid": m.group(1), "title": m.group(2),
                           "abstract": "", "mentions": []}
                continue
            m = re.match(r"^(\d+)\|a\|(.*)$", line)
            if m and current:
                current["abstract"] = m.group(2); continue
            parts = line.split("\t")
            if len(parts) >= 6 and current:
                try:
                    current["mentions"].append({
                        "start": int(parts[1]), "end": int(parts[2]),
                        "text": parts[3], "type": parts[4], "concept": parts[5]
                    })
                except ValueError:
                    pass
    if current: docs.append(current)
    return docs

mm_docs = parse_pubtator_gz(mm_path)
print(f"\nParsed {len(mm_docs):,} total documents")

# ── Assign splits using PMID files ────────────────────────────────────────────
mm_splits = {"train": [], "dev": [], "test": []}
for doc in mm_docs:
    for split, pmids in split_pmids.items():
        if doc["pmid"] in pmids:
            mm_splits[split].append(doc)
            break

print(f"\n  train: {len(mm_splits['train']):,} docs")
print(f"  dev  : {len(mm_splits['dev']):,} docs")
print(f"  test : {len(mm_splits['test']):,} docs")
print(f"\nFirst doc sample:")
print(json.dumps(mm_splits["train"][0], indent=2))

  train : 2,635 PMIDs
  dev   : 878 PMIDs
  test  : 879 PMIDs


Parsing: 365672it [00:01, 289408.13it/s]


Parsed 4,392 total documents

  train: 2,635 docs
  dev  : 878 docs
  test : 879 docs

First doc sample:
{
  "pmid": "25763772",
  "title": "DCTN4 as a modifier of chronic Pseudomonas aeruginosa infection in cystic fibrosis",
  "abstract": "Pseudomonas aeruginosa (Pa) infection in cystic fibrosis (CF) patients is associated with worse long-term pulmonary disease and shorter survival, and chronic Pa infection (CPA) is associated with reduced lung function, faster rate of lung decline, increased rates of exacerbations and shorter survival. By using exome sequencing and extreme phenotype design, it was recently shown that isoforms of dynactin 4 (DCTN4) may influence Pa infection in CF, leading to worse respiratory disease. The purpose of this study was to investigate the role of DCTN4 missense variants on Pa infection incidence, age at first Pa infection and chronic Pa infection incidence in a cohort of adult CF patients from a single centre. Polymerase chain reaction and direct sequenci

In [13]:
def parse_pubtator(file_path: str) -> list:
    """
    Parse PubTator format into a list of document dicts.
    Each dict: { pmid, title, abstract, full_text, mentions: [{start,end,text,type,concept_id}] }
    """
    documents = []
    current_doc = None

    # Handle both .txt and .gz
    opener = gzip.open if file_path.endswith(".gz") else open
    mode   = "rt" if file_path.endswith(".gz") else "r"

    with opener(file_path, mode, encoding="utf-8", errors="replace") as f:
        for line in tqdm(f, desc="Parsing MedMentions", mininterval=2.0):
            line = line.rstrip("\n")

            if not line.strip():
                # Blank line = end of document
                if current_doc is not None:
                    documents.append(current_doc)
                    current_doc = None
                continue

            # Title line:    PMID|t|text
            title_match = re.match(r"^(\d+)\|t\|(.*)$", line)
            if title_match:
                pmid, title = title_match.group(1), title_match.group(2)
                current_doc = {"pmid": pmid, "title": title,
                               "abstract": "", "full_text": "", "mentions": []}
                continue

            # Abstract line: PMID|a|text
            abstract_match = re.match(r"^(\d+)\|a\|(.*)$", line)
            if abstract_match and current_doc:
                current_doc["abstract"] = abstract_match.group(2)
                current_doc["full_text"] = current_doc["title"] + " " + current_doc["abstract"]
                continue

            # Annotation line: PMID\tStart\tEnd\tText\tType\tConceptID
            parts = line.split("\t")
            if len(parts) >= 6 and current_doc:
                try:
                    current_doc["mentions"].append({
                        "start":      int(parts[1]),
                        "end":        int(parts[2]),
                        "text":       parts[3],
                        "type":       parts[4],
                        "concept_id": parts[5],
                    })
                except (ValueError, IndexError):
                    pass

    # Catch last document if file doesn't end with blank line
    if current_doc is not None:
        documents.append(current_doc)

    return documents

if Path(mm_path).exists():
    mm_docs = parse_pubtator(mm_path)
    print(f"\nParsed {len(mm_docs):,} documents from MedMentions")
else:
    print(f"⚠️  File not found: {mm_path} — skipping MedMentions")
    mm_docs = []


Parsing MedMentions: 365672it [00:01, 288508.62it/s]


Parsed 4,392 documents from MedMentions


In [14]:
if mm_docs:
    print("=" * 60)
    print("MedMentions — DATA STRUCTURE")
    print("=" * 60)

    sample = mm_docs[0]
    print(f"\nPMID    : {sample['pmid']}")
    print(f"Title   : {sample['title'][:100]}")
    print(f"Abstract: {sample['abstract'][:200]}...")
    print(f"Mentions: {len(sample['mentions'])}")
    print("\nFirst 5 mentions:")
    for m in sample["mentions"][:5]:
        print(f"   [{m['start']}:{m['end']}] '{m['text']}' → type={m['type']}, concept={m['concept_id']}")

    # Entity type distribution
    print("\n" + "-" * 60)
    print("Entity type distribution (all documents):")
    all_types = Counter()
    total_mentions = 0
    for doc in mm_docs:
        for m in doc["mentions"]:
            all_types[m["type"]] += 1
            total_mentions += 1

    print(f"Total mentions: {total_mentions:,}")
    print(f"Unique types  : {len(all_types)}")
    for t, cnt in all_types.most_common(10):
        bar = "█" * (cnt * 30 // max(all_types.values()))
        print(f"   {t:20s} {cnt:6,}  {bar}")

    # Save processed version
    out_path = OUT_DIR / "medmentions_processed.json"
    with open(out_path, "w") as f:
        json.dump(mm_docs[:500], f, indent=2)  # Save first 500 docs as sample
    print(f"\nSaved 500-doc sample to {out_path}")
    print(f"   Full dataset: {len(mm_docs):,} documents, {total_mentions:,} mentions")

MedMentions — DATA STRUCTURE

PMID    : 25763772
Title   : DCTN4 as a modifier of chronic Pseudomonas aeruginosa infection in cystic fibrosis
Abstract: Pseudomonas aeruginosa (Pa) infection in cystic fibrosis (CF) patients is associated with worse long-term pulmonary disease and shorter survival, and chronic Pa infection (CPA) is associated with redu...
Mentions: 98

First 5 mentions:
   [0:5] 'DCTN4' → type=T116,T123, concept=C4308010
   [23:63] 'chronic Pseudomonas aeruginosa infection' → type=T047, concept=C0854135
   [67:82] 'cystic fibrosis' → type=T047, concept=C0010674
   [83:120] 'Pseudomonas aeruginosa (Pa) infection' → type=T047, concept=C0854135
   [124:139] 'cystic fibrosis' → type=T047, concept=C0010674

------------------------------------------------------------
Entity type distribution (all documents):
Total mentions: 352,496
Unique types  : 247
   T080                 31,485  ██████████████████████████████
   T169                 23,661  ██████████████████████
   T081 

In [15]:
import glob
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path

def parse_medquad_xml(xml_path: str) -> list:
    """Parse a single MedQuAD XML file into a list of QA dicts."""
    rows = []
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Get document-level metadata
        focus   = root.attrib.get("title", "")
        source  = root.attrib.get("source", Path(xml_path).parent.name)

        for qa in root.findall(".//QAPair"):
            question = qa.findtext("Question", default="").strip()
            answer   = qa.findtext("Answer",   default="").strip()
            qtype    = qa.find("Question").attrib.get("qtype", "") if qa.find("Question") is not None else ""

            if question:
                rows.append({
                    "question": question,
                    "answer":   answer,
                    "focus":    focus,
                    "qtype":    qtype,
                    "source":   source,
                    "file":     Path(xml_path).name,
                })
    except ET.ParseError as e:
        print(f"  ⚠️  XML parse error in {Path(xml_path).name}: {e}")
    return rows


# ── Load all 12 folders ────────────────────────────────────────────────────────
MEDQUAD_ROOT = "/content/MedQuAD"

xml_files = sorted(glob.glob(f"{MEDQUAD_ROOT}/**/*.xml", recursive=True))
print(f"Found {len(xml_files):,} XML files across 12 folders")

all_rows = []
folder_counts = {}

for xml_path in xml_files:
    folder = Path(xml_path).parent.name
    rows   = parse_medquad_xml(xml_path)
    all_rows.extend(rows)
    folder_counts[folder] = folder_counts.get(folder, 0) + len(rows)

mq_df = pd.DataFrame(all_rows)

print(f"\n✅ MedQuAD loaded: {len(mq_df):,} total QA pairs")
print(f"   With answers   : {mq_df['answer'].ne('').sum():,}")
print(f"   Unique qtypes  : {mq_df['qtype'].nunique()}")

# ── Per-folder breakdown ───────────────────────────────────────────────────────
print(f"\nQA pairs per source:")
for folder, count in sorted(folder_counts.items()):
    bar = "█" * (count * 30 // max(folder_counts.values()))
    print(f"  {folder:35s} {count:5,}  {bar}")

# ── Sample ─────────────────────────────────────────────────────────────────────
print(f"\nSample QA pair:")
sample = mq_df[mq_df["answer"].ne("")].iloc[0]
print(f"  Focus   : {sample['focus']}")
print(f"  QType   : {sample['qtype']}")
print(f"  Question: {sample['question']}")
print(f"  Answer  : {sample['answer'][:300]}...")

# ── Save ───────────────────────────────────────────────────────────────────────
out_path = "/content/medagent_data/medquad_processed.csv"
Path("/content/medagent_data").mkdir(exist_ok=True)
mq_df.to_csv(out_path, index=False)
print(f"\n✅ Saved to {out_path}")

Found 11,274 XML files across 12 folders

✅ MedQuAD loaded: 47,441 total QA pairs
   With answers   : 16,407
   Unique qtypes  : 39

QA pairs per source:
  10_MPlus_ADAM_QA                    17,348  ██████████████████████████████
  11_MPlusDrugs_QA                    12,889  ██████████████████████
  12_MPlusHerbsSupplements_QA           792  █
  1_CancerGov_QA                        729  █
  2_GARD_QA                           5,394  █████████
  3_GHR_QA                            5,430  █████████
  4_MPlus_Health_Topics_QA              981  █
  5_NIDDK_QA                          1,192  ██
  6_NINDS_QA                          1,088  █
  7_SeniorHealth_QA                     769  █
  8_NHLBI_QA_XML                        559  
  9_CDC_QA                              270  

Sample QA pair:
  Focus   : 
  QType   : information
  Question: What is (are) Adult Acute Lymphoblastic Leukemia ?
  Answer  : Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of

In [16]:
def load_medquad(path: str) -> pd.DataFrame:
    """
    Load MedQuAD from JSON or CSV into a standard DataFrame.
    Always returns columns: [question, answer, focus, qtype, source]
    """
    ext = Path(path).suffix.lower()

    if ext == ".json":
        with open(path) as f:
            data = json.load(f)

        # Handle list of dicts
        if isinstance(data, list):
            df = pd.DataFrame(data)
        # Handle dict with split keys
        elif isinstance(data, dict):
            # Could be {"train": [...], "test": [...]}
            if any(k in data for k in ["train", "test", "data"]):
                rows = []
                for split_data in data.values():
                    if isinstance(split_data, list):
                        rows.extend(split_data)
                df = pd.DataFrame(rows)
            else:
                df = pd.DataFrame([data])

    elif ext == ".csv":
        df = pd.read_csv(path)

    elif ext == ".parquet":
        df = pd.read_parquet(path)

    else:
        raise ValueError(f"Unsupported format: {ext}")

    # Normalize column names to standard set
    col_map = {}
    for col in df.columns:
        cl = col.lower()
        if "question" in cl and "question" not in col_map.values():
            col_map[col] = "question"
        elif "answer" in cl and "answer" not in col_map.values():
            col_map[col] = "answer"
        elif "focus" in cl or "topic" in cl or "disease" in cl:
            col_map[col] = "focus"
        elif "type" in cl and "qtype" not in col_map.values():
            col_map[col] = "qtype"
        elif "source" in cl or "url" in cl:
            col_map[col] = "source"

    df = df.rename(columns=col_map)

    # Ensure required columns exist
    for col in ["question", "answer", "focus", "qtype", "source"]:
        if col not in df.columns:
            df[col] = ""

    # Drop rows with no question or answer
    df = df.dropna(subset=["question"])
    df = df[df["question"].str.strip() != ""]

    return df[["question", "answer", "focus", "qtype", "source"]]



print(f"✅ MedQuAD loaded: {len(mq_df):,} rows")
print(f"   Columns : {list(mq_df.columns)}")

sample_row = mq_df[mq_df['answer'].ne('')].iloc[0]
print(f"\nFirst sample with answer:")
print(f"   Question : {sample_row['question']}")
print(f"   Answer   : {str(sample_row['answer'])[:300]}")
print(f"   Focus    : {sample_row['focus']}")
print(f"   QType    : {sample_row['qtype']}")
print(f"   Source   : {sample_row['source']}")

✅ MedQuAD loaded: 47,441 rows
   Columns : ['question', 'answer', 'focus', 'qtype', 'source', 'file']

First sample with answer:
   Question : What is (are) Adult Acute Lymphoblastic Leukemia ?
   Answer   : Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect red blood cells, white blood cells, and platelets.    - Previous chemotherapy and exposure to radia
   Focus    : 
   QType    : information
   Source   : CancerGov


In [17]:
if len(mq_df) > 0:
    print("=" * 60)
    print("MedQuAD — STATISTICS")
    print("=" * 60)

    print(f"\nTotal QA pairs  : {len(mq_df):,}")
    print(f"Unique questions: {mq_df['question'].nunique():,}")
    print(f"Has answers     : {mq_df['answer'].notna().sum():,} ({mq_df['answer'].notna().mean()*100:.0f}%)")

    # Answer length distribution
    mq_df["answer_len"] = mq_df["answer"].fillna("").str.split().str.len()
    print(f"\nAnswer length (words):")
    print(f"   Mean  : {mq_df['answer_len'].mean():.0f}")
    print(f"   Median: {mq_df['answer_len'].median():.0f}")
    print(f"   Max   : {mq_df['answer_len'].max():,}")

    # Question type distribution
    if mq_df["qtype"].notna().any() and mq_df["qtype"].str.strip().ne("").any():
        print(f"\nTop question types:")
        for qt, cnt in mq_df["qtype"].value_counts().head(8).items():
            bar = "█" * (cnt * 25 // mq_df["qtype"].value_counts().iloc[0])
            print(f"   {str(qt):30s} {cnt:5,}  {bar}")

    # Save
    out_path = OUT_DIR / "medquad_processed.csv"
    mq_df.to_csv(out_path, index=False)
    print(f"\n✅ Saved to {out_path}")

MedQuAD — STATISTICS

Total QA pairs  : 47,441
Unique questions: 44,603
Has answers     : 47,441 (100%)

Answer length (words):
   Mean  : 70
   Median: 0
   Max   : 4,281

Top question types:
   information                    9,214  █████████████████████████
   symptoms                       4,338  ███████████
   treatment                      3,906  ██████████
   causes                         2,436  ██████
   outlook                        2,232  ██████
   exams and tests                2,058  █████
   when to contact a medical professional 1,738  ████
   inheritance                    1,446  ███

✅ Saved to /content/medagent_data/medquad_processed.csv


In [18]:
# ── Find DrugBank file ────────────────────────────────────────────────────────
db_candidates = [

    "/content/drugbank vocabulary.csv",

]

db_path = next((p for p in db_candidates if Path(p).exists()), None)

# Also check for any tsv/csv file
if db_path is None:
    tsv_files = list(Path("/content").glob("*.tsv"))
    if tsv_files:
        db_path = str(tsv_files[0])
        print(f"Auto-detected: {db_path}")

if db_path and Path(db_path).exists():
    sep = "\t" if db_path.endswith(".tsv") else ","
    try:
        drugbank_df = pd.read_csv(db_path, sep=sep, on_bad_lines="skip")
        print(f"✅ DrugBank loaded: {len(drugbank_df):,} rows")
        print(f"   Columns: {list(drugbank_df.columns)}")
        print(f"\nFirst 3 rows:")
        print(drugbank_df.head(3).to_string())
    except Exception as e:
        print(f"⚠️  Failed to load DrugBank: {e}")
        drugbank_df = pd.DataFrame()
else:
    print("⚠️  DrugBank file not found.")
    print("   Downloading open-data TSV from GitHub mirror...")
    try:
        drugbank_df = pd.read_csv(
            "https://raw.githubusercontent.com/dhimmel/drugbank/gh-pages/data/drugbank.tsv",
            sep="\t"
        )
        print(f"✅ Downloaded DrugBank: {len(drugbank_df):,} rows")
        drugbank_df.to_csv(OUT_DIR / "drugbank.tsv", sep="\t", index=False)
    except Exception as e:
        print(f"   Download failed: {e}")

        drugbank_df = pd.DataFrame()

✅ DrugBank loaded: 19,842 rows
   Columns: ['DrugBank ID', 'Accession Numbers', 'Common name', 'CAS', 'UNII', 'Synonyms', 'Standard InChI Key']

First 3 rows:
  DrugBank ID     Accession Numbers   Common name          CAS        UNII                                                                                                                                                               Synonyms Standard InChI Key
0     DB00001  BIOD00024 | BTD00024     Lepirudin  138068-37-8  Y43GF64R34                                                  [Leu1, Thr2]-63-desulfohirudin | Desulfatohirudin | Hirudin variant-1 | Lepirudin | Lepirudin recombinant | R-hirudin                NaN
1     DB00002  BIOD00071 | BTD00071     Cetuximab  205923-56-4  PQX0D8J21J                                                                           Cetuximab | Cétuximab | Cetuximabum | Chimeric MoAb C225 | Chimeric Monoclonal Antibody C225                NaN
2     DB00003  BIOD00001 | BTD00001  Dornase alfa  143831-

In [19]:
if len(drugbank_df) > 0:
    print("=" * 60)
    print("DRUGBANK — STATISTICS")
    print("=" * 60)

    print(f"\nTotal drugs: {len(drugbank_df):,}")
    print(f"Columns    : {list(drugbank_df.columns)}")

    # Show column value counts for key fields
    for col in drugbank_df.columns[:8]:
        n_unique = drugbank_df[col].nunique()
        n_null   = drugbank_df[col].isna().sum()
        print(f"   {col:30s}  unique={n_unique:,}  null={n_null:,}")

    # Save
    out_path = OUT_DIR / "drugbank_processed.csv"
    drugbank_df.to_csv(out_path, index=False)
    print(f"\n✅ Saved to {out_path}")

DRUGBANK — STATISTICS

Total drugs: 19,842
Columns    : ['DrugBank ID', 'Accession Numbers', 'Common name', 'CAS', 'UNII', 'Synonyms', 'Standard InChI Key']
   DrugBank ID                     unique=19,842  null=0
   Accession Numbers               unique=4,661  null=15,181
   Common name                     unique=19,842  null=0
   CAS                             unique=13,080  null=6,735
   UNII                            unique=15,183  null=4,659
   Synonyms                        unique=18,160  null=1,682
   Standard InChI Key              unique=14,614  null=5,220

✅ Saved to /content/medagent_data/drugbank_processed.csv


In [20]:
import re
import pandas as pd
from pathlib import Path

def parse_ncbi_inline(file_path: str) -> list:
    """
    Parse NCBI Disease corpus - inline XML category tags format.
    Each line: PMID\tTitle (with tags)\tAbstract (with tags)
    Tags look like: <category="SpecificDisease">mention text</category>
    """
    documents = []
    tag_pattern = re.compile(r'<category="([^"]+)">([^<]+)</category>')

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split("\t", 2)
            if len(parts) < 3:
                continue

            pmid, title_raw, abstract_raw = parts[0], parts[1], parts[2]

            # Extract all mentions from title + abstract
            mentions = []
            full_raw = title_raw + " " + abstract_raw

            for match in tag_pattern.finditer(full_raw):
                mentions.append({
                    "text":  match.group(2),
                    "type":  match.group(1),
                    "start": match.start(),
                    "end":   match.end(),
                })

            # Strip tags to get clean text
            clean_title    = tag_pattern.sub(r'\2', title_raw).strip()
            clean_abstract = tag_pattern.sub(r'\2', abstract_raw).strip()

            documents.append({
                "pmid":     pmid,
                "title":    clean_title,
                "abstract": clean_abstract,
                "full_text": clean_title + " " + clean_abstract,
                "mentions": mentions,
            })

    return documents


# ── Load all 3 splits ──────────────────────────────────────────────────────────
ncbi_splits = {
    "train": parse_ncbi_inline("/content/NCBI_corpus_training.txt"),
    "dev":   parse_ncbi_inline("/content/NCBI_corpus_development.txt"),
    "test":  parse_ncbi_inline("/content/NCBI_corpus_testing.txt"),
}

print("✅ NCBI Disease loaded")
for split, docs in ncbi_splits.items():
    total_mentions = sum(len(d["mentions"]) for d in docs)
    print(f"   {split:6s}: {len(docs):,} docs  |  {total_mentions:,} mentions")

# ── Sample ─────────────────────────────────────────────────────────────────────
sample = ncbi_splits["train"][0]
print(f"\nSample:")
print(f"  PMID    : {sample['pmid']}")
print(f"  Title   : {sample['title'][:80]}")
print(f"  Mentions: {sample['mentions'][:3]}")

# ── Entity type breakdown ──────────────────────────────────────────────────────
from collections import Counter
all_types = Counter()
for split_docs in ncbi_splits.values():
    for doc in split_docs:
        for m in doc["mentions"]:
            all_types[m["type"]] += 1

print(f"\nEntity types:")
for t, cnt in all_types.most_common():
    bar = "█" * (cnt * 25 // max(all_types.values()))
    print(f"   {t:25s} {cnt:5,}  {bar}")

✅ NCBI Disease loaded
   train : 593 docs  |  5,148 mentions
   dev   : 100 docs  |  791 mentions
   test  : 100 docs  |  961 mentions

Sample:
  PMID    : 10021369
  Title   : Identification of APC2, a homologue of the adenomatous polyposis coli tumour sup
  Mentions: [{'text': 'adenomatous polyposis coli tumour', 'type': 'Modifier', 'start': 43, 'end': 108}, {'text': 'adenomatous polyposis coli ( APC ) tumour', 'type': 'Modifier', 'start': 126, 'end': 199}, {'text': 'colon carcinoma', 'type': 'Modifier', 'start': 431, 'end': 478}]

Entity types:
   SpecificDisease           3,924  █████████████████████████
   Modifier                  1,774  ███████████
   DiseaseClass              1,029  ██████
   CompositeMention            173  █


In [22]:
print("=" * 65)
print("MEDAGENT — DATASET READINESS REPORT")
print("=" * 65)

checks = [
    (
        "BC5CDR (NER)",
        len(bc5cdr.get("train", [])) > 0,
        f"train={len(bc5cdr.get('train',[]))} / test={len(bc5cdr.get('test',[]))} / valid={len(bc5cdr.get('valid',[]))}"
    ),
    (
        "MedMentions (Rich NER)",
        len(mm_docs) > 0,
        f"{len(mm_docs):,} documents" if mm_docs else "not loaded"
    ),
    (
        "MedQuAD (QA + Simplification)",
        len(mq_df) > 0,
        f"{len(mq_df):,} QA pairs"
    ),
    (
        "NCBI Disease (Classification)",
        ncbi_splits is not None,
        f"train={len(ncbi_splits['train'])} / test={len(ncbi_splits['test'])}" if ncbi_splits else "not loaded"
    ),
    (
        "DrugBank (Drug NLG)",
        len(drugbank_df) > 0,
        f"{len(drugbank_df):,} drugs" if len(drugbank_df) > 0 else "not loaded"
    ),
]

all_ok = True
for name, ok, detail in checks:
    icon = "✅" if ok else "❌"
    print(f"  {icon}  {name:35s}  {detail}")
    if not ok:
        all_ok = False

print("=" * 65)
if all_ok:
    print("  🎉 All datasets loaded. Ready for Module 02 — Medical NER.")
else:
    print("  ⚠️  Some datasets missing. Check warnings above.")
    print("  You can still proceed — missing datasets can be added later.")

print("\n  Saved files:")
for f in sorted(OUT_DIR.glob("*")):
    print(f"    {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

MEDAGENT — DATASET READINESS REPORT
  ✅  BC5CDR (NER)                         train=5228 / test=5865 / valid=5330
  ✅  MedMentions (Rich NER)               4,392 documents
  ✅  MedQuAD (QA + Simplification)        47,441 QA pairs
  ✅  NCBI Disease (Classification)        train=593 / test=100
  ✅  DrugBank (Drug NLG)                  19,842 drugs
  🎉 All datasets loaded. Ready for Module 02 — Medical NER.

  Saved files:
    bc5cdr_meta.json  (0 KB)
    drugbank_processed.csv  (3041 KB)
    medmentions_processed.json  (7297 KB)
    medquad_processed.csv  (25071 KB)
